In [1]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this from the users location.",
                    },
                },
                "required": ["location", "format"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_n_day_weather_forecast",
            "description": "Get an N-day weather forecast",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this from the users location.",
                    },
                    "num_days": {
                        "type": "integer",
                        "description": "The number of days to forecast",
                    },
                },
                "required": ["location", "format", "num_days"],
            },
        },
    },
]

In [2]:
import requests
import os

resp = requests.post(
    "https://api.dartmouth.edu/api/jwt",
    headers={"Authorization": os.environ["DARTMOUTH_API_KEY"]},
)
jwt = resp.json()["jwt"]

In [3]:
from openai import OpenAI

# Initialize the client, pointing it to one of the available models
client = OpenAI(
    base_url="https://ai-api.dartmouth.edu/tgi/llama-3-1-8b-instruct/v1",
    api_key="_",
    default_headers={"Authorization": f"Bearer {jwt}"},
)

chat_completion = client.chat.completions.create(
    model="tgi",
    messages=[
        {
            "role": "system",
            "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.",
        },
        {
            "role": "user",
            "content": "What's the weather like the next 3 days in San Francisco, CA?",
        },
    ],
    tools=tools,
    tool_choice="auto",  # tool selected by model
    max_tokens=500,
)


called = chat_completion.choices[0].message.tool_calls
print(called)

[ChatCompletionMessageToolCall(id='0', function=Function(arguments={'format': 'celsius', 'location': 'San Francisco, CA', 'num_days': 3}, name='get_n_day_weather_forecast', description=None), type='function')]


In [4]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    openai_api_base="https://ai-api.dartmouth.edu/tgi/llama-3-1-8b-instruct/v1",
)
llm.client._client._custom_headers.update({"Authorization": f"Bearer {jwt}"})

llm.invoke("Hello!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 37, 'total_tokens': 47, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': '2.3.1-sha-a094729', 'finish_reason': 'stop', 'logprobs': None}, id='run-52f2326f-5a9f-4595-83b6-06d60f8eeadd-0', usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b


tools = [add, multiply]

In [6]:
llm_with_tools = llm.bind_tools(tools)

In [7]:
query = "What is 10 plus 5 times 5?"

llm_with_tools.invoke(query)

/Users/f006pfk/source/langchain-dartmouth-cookbook/.venv/lib/python3.12/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


ValidationError: 1 validation error for AIMessage
invalid_tool_calls.0.args
  Input should be a valid string [type=string_type, input_value={'a': 10, 'b': 5}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/string_type